In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from expand_subgraph import ExpandSubgraph
from model_2 import GNN_auto, Projector
import pickle as pkl
import numpy as np
import torch
from load_data import DataLoader
# from loader2 import DataLoader2 as GnnDataLoader
from reasoning_module2_ import ReasoningModule2 as ReasoningModule
import time
import random

In [3]:
from data_utils import id2ent, ent2name, id2rel, rel2id

In [4]:
with open("data_for_CL/llm_description.pkl", "rb") as f:
    llm_description = pkl.load(f)

In [5]:
with open("knowledge_graph/queries/CWQ_sim_queries2.pkl", "rb") as f:
    queries = pkl.load(f)
id_query = 222
queries[id_query]

{'query_type': ('e', ('r', 'r', 'r')),
 'raw_query': (10363, (398, 255, 49)),
 'named_query': ('Codemasters',
  ('+/organization/organization/place_founded',
   '-/base/biblioness/bibs_location/state',
   '-/people/person/place_of_birth')),
 'transformed_query': 'In which location was William Shakespeare born, related to the founding place of Codemasters?',
 'answers_id': [4732],
 'answers': ['William_Shakespeare'],
 'natural_query': 'In which location was William Shakespeare born, related to the founding place of Codemasters?',
 'gold_path': [[8547, 254, 11490], [10363, 398, 11490], [4732, 48, 8547]],
 '1_hop_edges': []}

#### Examples:
Example 1: 153  
Example 2: 78  
Example 3: 159
Example 4: 161

In [6]:
class Config:
    data_path = 'knowledge_graph/KG_data/FB15k-237-betae'
    seed = 1234
    k_rel = 4 # beams
    k_cands = 120
    depth = 2 # max depth of subgraph
    cands_lim = 1024
    gpu = 0
    fact_ratio = 1.0
    val_num = -1 # how many triples are used as the validate set
    add_manual_edges = False
    remove_1hop_edges = True
    not_shuffle_train = False
    device = "cuda:0"

In [7]:
args = Config()
loader = DataLoader(args, mode='train')

# loader.shuffle_train()
train_graph = loader.train_graph
train_graph_homo = list(set([(h,t) for (h,r,t) in train_graph]))

args.n_ent = loader.n_ent
args.n_rel = loader.n_rel

544230 0
==> removing 1-hop links...
==> done


In [8]:
GoG_args = {
    'drop_ratio': 0.4,
}

sampler = ExpandSubgraph(
    args.n_ent, args.n_rel, train_graph_homo, train_graph, args,
    GoG_simulation=True,
    GoG_args=GoG_args
)

In [9]:
sampler.assign_query(queries[id_query])
subgraph_data = sampler.sampleSubgraph()

Simulating GoG by removing edges...


In [10]:
print(len(sampler.orignal_edge_index), len(sampler.edge_index))

544230 544226


In [11]:
def check_reachability(id_query, subgraph_data):
    summ = 0
    for ans in queries[id_query]['answers_id']:
        cnt = 0
        for edges in subgraph_data[2]:
            if ans in edges[[0,2]]:
                cnt += 1

        summ += cnt
        # print(ans, cnt)
        # print(summ)
        return summ

In [12]:
args.n_rel=237

In [13]:
class GNN_config:
    n_layer = 3
    hidden_dim = 64
    attn_dim = 4
    dropout = 0.3
    n_ent = 1024 # subgraph size
    n_rel = 237
    train_ver = 20
    shortcut = True
    readout = 'linear'
    concatHidden = True
    initializer = 'relation'
    
class Projector_config:
    n_layers = 4
    in_dim = 4096
    hidden_dims = [512, 256]
    out_dim = 64

In [14]:
modelPath = "weights/finetune/topk_ValMRR_0.269.pt"
checkpoint = torch.load(modelPath, map_location=torch.device("cuda:0"), weights_only=False)


    # llm_description_emb_path = "data_for_CL/llm_description_aligned_emb.pkl"
    # ## 474 relations / 20 versions each / 4096 dimensional
    # with open(llm_description_emb_path, "rb") as f:
    #     llm_description_aligned_emb = pkl.load(f)
    # llm_emb = list(llm_description_aligned_emb.values())
    # llm_emb = torch.stack(llm_emb, dim=0)
    # pretrain_model_path = "weights/finetune/avid-forest-122.pt"
    # del(llm_description_aligned_emb)


gnn_model = GNN_auto(GNN_config, loader)
gnn_model.load_state_dict(checkpoint['gnn_model'])
gnn_model = gnn_model.to(torch.device("cuda:0"))

In [15]:


projector_model = Projector(Projector_config.in_dim, Projector_config.hidden_dims, Projector_config.out_dim, Projector_config.n_layers)
projector_model.load_state_dict(checkpoint['projector'])
projector_model = projector_model.to(torch.device("cuda:0"))

In [16]:
reasoning_module = ReasoningModule(sampler,
                                   gnn_model,
                                   projector_model,
                                   "gpt-3.5-turbo") 
# gpt-4o, gpt-3.5-turbo, "meta-llama/Llama-3.1-8B-Instruct"


In [17]:
reasoning_module.assign_query(queries[id_query])


Simulating GoG by removing edges...


In [18]:

def create_dicts(subgraph_data):
    ent_dict = {}
    for idx in subgraph_data[0].cpu().numpy():
        ent_dict[idx.item()] = ent2name[id2ent[idx]]

    rel_dict = {}
    rel_desc_dict = {}
    rel_tensor = subgraph_data[2][:,1]
    unique_rel, rel_local = torch.unique(rel_tensor, return_inverse=True)

    for idx in rel_local:
        raw_rel = id2rel[unique_rel[idx].item()]
        rel_nl = raw_rel.split("/")
        # print(rel_nl)
        if rel_nl[0] == "+":
            rel_dict[idx.item()] = "/".join(rel_nl[1:])  # Remove the leading "+"
            
        else:
            rel_dict[idx.item()] = "/".join(rel_nl[1:][:-1])  # Join back with "/"
        # rel_dict[idx.item()] = id2rel[idx]
        rel_desc_dict[idx.item()] = llm_description.get(raw_rel, "No description available")[0]
    return ent_dict, rel_dict, rel_desc_dict

In [19]:
subgraph_data = reasoning_module.subgraph_data
ent_dict, rel_dict, rel_desc_dict = create_dicts(subgraph_data)

In [20]:
# subgraph_edges = subgraph_data[2]
# for edges in subgraph_edges:
#     inv_rel_idx = edges[1] + 1 if edges[1] % 2 == 0 else edges[1] - 1
#     inv_edge = torch.tensor([edges[2], inv_rel_idx, edges[0]])
#     # Check if inv_edge in subgraph_edges
#     if any((subgraph_edges[:,0] == inv_edge[0]) & (subgraph_edges[:,1] == inv_edge[1]) & (subgraph_edges[:,2] == inv_edge[2])):
#         # print(f"Missing inverse edge for {edges}: {inv_edge}")
#         assert False, f"exist inverse edge for {edges}: {inv_edge}"

In [21]:
reasoning_module.assign_dict(ent_dict, rel_dict, rel_desc_dict)

In [22]:
queries[id_query]

{'query_type': ('e', ('r', 'r', 'r')),
 'raw_query': (10363, (398, 255, 49)),
 'named_query': ('Codemasters',
  ('+/organization/organization/place_founded',
   '-/base/biblioness/bibs_location/state',
   '-/people/person/place_of_birth')),
 'transformed_query': 'In which location was William Shakespeare born, related to the founding place of Codemasters?',
 'answers_id': [4732],
 'answers': ['William_Shakespeare'],
 'natural_query': 'In which location was William Shakespeare born, related to the founding place of Codemasters?',
 'gold_path': [[8547, 254, 11490], [10363, 398, 11490], [4732, 48, 8547]],
 '1_hop_edges': []}

In [23]:
# for idx in range(len(unique_rel)):
#     print(idx, rel_dict[idx],"||||", rel_desc_dict[idx])

In [24]:
print(queries[id_query]['natural_query'])
print(queries[id_query][r'named_query'])
check_reachability(id_query, reasoning_module.subgraph_data)

In which location was William Shakespeare born, related to the founding place of Codemasters?
('Codemasters', ('+/organization/organization/place_founded', '-/base/biblioness/bibs_location/state', '-/people/person/place_of_birth'))


0

In [25]:
# llm_description["-/film/film_distributor/films_distributed./film/film_film_distributor_relationship/film"]

In [26]:
reasoning_module.reasoning()

Hop 1 with 1 active path(s)
path: 0 {'current_entity': 10363, 'trace': []}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
3 new paths generated at hop 1
Hop 1: 3 active path(s)
Hop 2 with 3 active path(s)
path: 0 {'current_entity': 32, 'trace': [(10363, 'Specifies the birth location of an individual.', 32)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
path: 1 {'current_entity': 119, 'trace': [(10363, 'Specifies the birth location of an individual.', 119)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
path: 2 {'current_entity': 10423, 'trace': [(10363, 'Specifies the birth location of an individual.', 10423)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
12 new paths generated at hop 2
Hop 2: 8 active path(s)
Hop 3 with 8 active path(s)
path: 0 {'current_entity': 9094, 'trace': [(10363, 'Specifies the birth location of an 

## Run

In [28]:
subgraph_reachability = []
results = []
ground_truths = []
for idx, query in enumerate(queries[:2]):
    reasoning_module.assign_query(query)
    subgraph_data = reasoning_module.subgraph_data
    ent_dict, rel_dict, rel_desc_dict = create_dicts(subgraph_data)
    is_reachable = check_reachability(idx, subgraph_data)
    subgraph_reachability.append(is_reachable)
    if is_reachable == 0:
        results.append(None)
        ground_truths.append(query['answers_id'])
        continue
    reasoning_module.assign_dict(ent_dict, rel_dict, rel_desc_dict)
    reasoning_module.reasoning()
    results.append(reasoning_module.result)
    ground_truths.append(query['answers_id'])
    time.sleep(1)

Simulating GoG by removing edges...
Simulating GoG by removing edges...
Hop 1 with 1 active path(s)
path: 0 {'current_entity': 6290, 'trace': []}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
4 new paths generated at hop 1
Hop 1: 4 active path(s)
Hop 2 with 4 active path(s)
path: 0 {'current_entity': 10176, 'trace': [(6290, 'Identifies the films in which the actor has performed.', 10176)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
------ call_llm_filter:
path: 1 {'current_entity': 2125, 'trace': [(6290, 'Identifies the films in which the actor has performed.', 2125)]}
------ call_llm_judge:
------ call_llm_pathfinder:
------ call_llm_filter:
path: 2 {'current_entity': 1431, 'trace': [(6290, 'Identifies the TV programs in which the actor had a regular cast role', 1431)]}
------ call_llm_judge:
Answer found: {YES}. Answer: ["The Office"]
